In [2]:
%load_ext cuml.accel
%run /workspace/alvin/SAR_ML/notebooks/SSR/SSRtransforms_preloaded.py
# %run /mnt/d/Users/Admin/Projects/dso/SAR_ML/notebooks/SSR/SSRtransforms_preloaded.py
import os
import random
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, ConcatDataset, Subset, Dataset
import re
import copy
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm import tqdm

/opt/py_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Uncomment only if you need 100% determinism and can handle errors
    # torch.use_deterministic_algorithms(True, warn_only=True)
    # os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    os.environ["PYTHONHASHSEED"] = str(seed)

def worker_init_fn(worker_id):
    """DataLoader worker init for reproducibility"""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [7]:
workspace = "/workspace/alvin/SAR_ML"
# workspace = "/mnt/d/Users/Admin/Projects/dso/SAR_ML"
data_workspace = os.path.join(workspace, "data/SAMPLE")

In [8]:
gmm_cache = build_gmm_cache(input_dir = os.path.join(data_workspace, "mat_files/synth"), processing_func = LogMapping(c = 1000.0))

Found 1345 .mat files


Fitting GMMs: 100%|███████████████████████████████████████████████████████████████████| 1345/1345 [01:00<00:00, 22.05it/s]


In [9]:
synth_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/synth"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), SSRAugmentation(gmm_cache, alpha=0.6, beta=0.4, apply_prob=0.5, gaussian_noise = True, mu_s = 0.0, sigma_s = 0.3), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

meas_ds = DatasetFolderWithPath(
    os.path.join(data_workspace, "mat_files/real"),
    extensions = (".mat"),
    transform = transforms.Compose([Magnitude(), LogMapping(c = 1000.0), NumpyToTensor3Channel()]),
    loader = mat_file_loader
)

In [10]:
excluded_label = 0
excluded_label_name = synth_ds.classes[excluded_label]
print(f"Excluding label {excluded_label} ({excluded_label_name}) from synthetic dataset")
new_synth_ds = RemappedSubset(synth_ds, exclude_label=excluded_label)

Excluding label 0 (2s1) from synthetic dataset


In [11]:
train_ds = new_synth_ds
test_ds = meas_ds

ds_dict = {"train" : train_ds, "test": test_ds}
dataset_sizes = {"train" : len(train_ds), "test": len(test_ds)}

In [75]:
seed_lst = [42]

label_lst = []
prob_lst = []
pred_lst = []

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
for i, seed in enumerate(seed_lst):
    print(f"Evaluating Run {i}: seed {seed}")
    new_model = models.resnet18(weights = None) # dont load ImageNet Weights
    new_model.fc = nn.Sequential(
        nn.Dropout(p = 0.4),
        nn.Linear(new_model.fc.in_features, len(synth_ds.class_to_idx) - 1)
    )
    
    # Load your trained weights
    new_model.load_state_dict(torch.load(
        os.path.join(workspace, f"weights/SSR/OOD/rn18_seed{seed}_b16_rm_{excluded_label_name}.pth"),
        map_location=device
    ))
    
    new_model = new_model.to(device)
    new_model.eval()
    
    with torch.no_grad():
        for inputs, labels in DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=12, pin_memory=True, persistent_workers=True):
            inputs = inputs.to(device)
    
            outputs = new_model(inputs)
            probs = F.softmax(outputs, dim=1)
            confidence, preds = torch.max(probs, dim=1)
    
            prob_lst.append(confidence.cpu())
            pred_lst.append(preds.cpu())
            label_lst.append(labels)  # already on cpu from dataloader

confidences = torch.cat(prob_lst).numpy()
preds = torch.cat(pred_lst).numpy()
labels = torch.cat(label_lst).numpy()

label_map = new_synth_ds.label_map  # {1:0, 2:1, ..., 9:8}

# remap ID labels, leave OOD (excluded_label=0) unchanged
remapped_labels = np.array([
    label_map[l] if l in label_map else l  # OOD label (0) stays as 0
    for l in labels
])

Evaluating Run 0: seed 42


In [92]:
remapped_labels

array([0, 0, 0, ..., 8, 8, 8], shape=(1345,))

In [94]:
preds

array([3, 3, 3, ..., 8, 8, 8], shape=(1345,))

In [93]:
confidences

array([0.8616474 , 0.998623  , 0.952683  , ..., 0.99997926, 0.8673863 ,
       0.99911493], shape=(1345,), dtype=float32)

In [108]:
synth_ds.classes

['2s1', 'bmp2', 'btr70', 'm1', 'm2', 'm35', 'm548', 'm60', 't72', 'zsu23']

In [109]:
new_synth_ds.label_map

{1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8}

In [111]:
[synth_ds.classes[i] for i,j in new_synth_ds.label_map.items()]

['bmp2', 'btr70', 'm1', 'm2', 'm35', 'm548', 'm60', 't72', 'zsu23']

In [99]:
preds[remapped_labels == 0]

array([3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3,
       0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 8, 8, 8, 1, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 3, 3, 1, 3, 3, 3, 1, 3, 1,
       1, 3, 1, 3, 3, 3, 1, 1, 3, 1, 3, 1, 3, 1, 3, 1, 3, 3, 3, 3, 1, 3,
       3, 3, 3, 3, 3, 3, 1, 3, 3, 0, 8, 8, 8, 0, 0, 1, 3, 3, 0, 0, 0, 0,
       1, 0, 0, 0, 1, 3, 1, 3, 3, 3, 1, 1, 1, 3, 3, 1, 3, 1, 1, 3, 1, 3,
       1, 1, 1, 1, 1, 3, 3, 1, 3, 1, 3, 1, 1, 1, 1, 3, 3, 3, 3, 3, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [100]:
confidences[remapped_labels == 0]

array([0.8616474 , 0.998623  , 0.952683  , 0.99976367, 0.99615747,
       0.99859947, 0.9902621 , 0.99515605, 0.9503642 , 0.99926764,
       0.933996  , 0.9987846 , 0.99302   , 0.979871  , 0.72620845,
       0.99969685, 0.99241155, 0.9935128 , 0.7415103 , 0.9976847 ,
       0.9700523 , 0.99695194, 0.98691803, 0.94421   , 0.9966215 ,
       0.99965966, 0.99824286, 0.99554825, 0.9907403 , 0.8129247 ,
       0.9805555 , 0.9592645 , 0.9339058 , 0.5944319 , 0.83766645,
       0.99927384, 0.999423  , 0.55735797, 0.6162716 , 0.9553565 ,
       0.9807114 , 0.99093366, 0.98424226, 0.9982492 , 0.9430589 ,
       0.9997571 , 0.99970263, 0.9673547 , 0.9985783 , 0.9931005 ,
       0.92759633, 0.8357859 , 0.9291987 , 0.98301965, 0.89228314,
       0.99757355, 0.495936  , 0.49964708, 0.6129614 , 0.5467546 ,
       0.4787485 , 0.7009572 , 0.997775  , 0.9966577 , 0.9548415 ,
       0.8983214 , 0.48464614, 0.95115805, 0.8183656 , 0.8793775 ,
       0.9727883 , 0.92143625, 0.31691772, 0.97597504, 0.99947

In [105]:
confidences[(remapped_labels == 0) & (preds == 0)]

array([0.99241155, 0.98691803, 0.9622561 , 0.94878924, 0.8617367 ,
       0.9287762 , 0.9099658 , 0.8499446 , 0.87320834, 0.89518446,
       0.997479  , 0.5275177 , 0.49372187, 0.9996561 , 0.99995005,
       0.99995315, 0.99988294, 0.9999802 , 0.99996424, 0.9999813 ,
       0.99992657, 0.99950457, 0.9999864 , 0.9999814 , 0.9999331 ,
       0.9986492 , 0.9999181 , 0.99962366, 0.9998136 , 0.9999833 ,
       0.99998224, 0.999856  , 0.99924856, 0.99997616, 0.9999486 ,
       0.9999064 , 0.9999287 , 0.99738175, 0.9995203 , 0.9999777 ,
       0.9993923 , 0.9998995 , 0.9998759 , 0.99994683, 0.9999902 ,
       0.9999639 , 0.9999306 , 0.9999434 , 0.9999522 , 0.9999763 ,
       0.9999498 , 0.99993765, 0.99997675, 0.9999776 , 0.9999367 ,
       0.99980706, 0.9998795 , 0.99997437, 0.99997294, 0.9993574 ,
       0.9999279 , 0.99989605, 0.9999418 , 0.99983954, 0.9990415 ,
       0.9295423 , 0.99819475, 0.9995974 , 0.99986744, 0.9994821 ,
       0.9976145 , 0.9999697 , 0.99991894, 0.99993694, 0.99991